# MAPED: automatic processing on MPS

Load your tilt series and run the complete alignment and merge with one call.

## Setup

**Development prerequisite:** this workflow still requires MAPED development builds; the package command alone does not yet provide these APIs.

New to Python? Follow the [Conda or uv setup guide](https://github.com/bobleesj/quantem-tutorials/blob/oct-17-maped-tutorials/tutorials/diffraction/maped/setup/README.md). If packages are missing in an existing Python 3.11+ environment, uncomment the cell below, then restart the kernel.

In [ ]:
# %pip install --upgrade --pre -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "quantem.widget[mps]" quantem pandas

In [ ]:
from pathlib import Path

import quantem.widget as qw
from quantem.diffraction import MAPED
from quantem.gpu import io

folder = Path("tilt-series")
output = Path("outputs/merged_automatic_master.h5")
device = "mps"
qw.profile()

## Load data

Put one Arina tilt series in `folder`, including master, data, and metadata files. Loading applies median replacement at stored bad detector pixels. Check that this correction and the recorded calibration are appropriate for your data.

In [ ]:
files = io.discover(folder)
tilts = io.load(
    files, device=device, apply_mask=False,
    hot_pixel_correction="median",
)
maped = MAPED.from_datasets(tilts)
maped.show_tilts(num_cols=7, size="medium")

## Align and merge

`run()` finds the beam centers, calls `align_diffraction()` and `align_images()`, and merges the common field of view. These are the same stages as the step-by-step MPS notebook.

Optionally pass `images=images` to `run()` for BF, ADF, DF or simultaneously recorded HAADF images: one 2D array per tilt on the same scan grid, in tilt order. Merging still uses the original diffraction data.

Save as scaled uint16 HDF5 with measured rounding error. Reopen as ANS-resident data and decode only requested regions.

In [ ]:
output.parent.mkdir(parents=True, exist_ok=True)
merged = maped.run(iterations=160, crop=True, dtype="scaled_uint16", save_to=output, release_tilts=True, show=False)
maped.report()  # Table; use as_dict=True for a dictionary.

In [ ]:
maped.show_alignment(num_cols=7, size="medium", reference=0)  # Set the zero-tilt index.

## Inspect results

Inspect merged diffraction patterns at three marked scan positions. Input tilts have been released to save memory; the alignment view uses retained 2D images.

In [ ]:
maped.show(positions=[(128, 128), (256, 256), (384, 384)], tilt_indices=[])  # Merged patterns; inputs were released.

In [ ]:
qw.Show4DSTEM(merged, device=device)

## Reopen (optional)

The HDF5 result is already saved with its calibration and alignment metadata. Reopening restores calibrated intensities; it does not recover the original float32 values lost to uint16 rounding.

In [ ]:
# reopened = io.load(output, device=device)
# qw.Show4DSTEM(reopened, device=device)

Built on Henry Bell’s [QuantEM PR #256](https://github.com/electronmicroscopy/quantem/pull/256). Experimental inputs and executed outputs are not included in this shareable template.